<a href="https://colab.research.google.com/github/apcyssr/2026_Spring_KBU/blob/main/20260417_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NLP Basics: Newsgroup Classification using CountVectorizer**

#**1. Task Overview**

The goal of this assignment is to utilize the sklearn library's 20 Newsgroups dataset to convert text data into numerical vectors and classify new sentences based on their Cosine Similarity to existing topics.

#**2. Environment Setup**

We will import the necessary libraries for dataset handling, text vectorization, and similarity calculation.

In [2]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#**3. Step 1: Data Loading and Sampling**

We select three specific categories: Graphics, Space, and Religion. To reduce noise, we remove headers, footers, and quotes.Note: We intentionally extract only 20 samples per category to observe how data scarcity affects similarity scores (potentially resulting in 0.0 similarity).

In [3]:
# Define categories
categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']

# Load newsgroups data
newsgroups = fetch_20newsgroups(subset='train', categories=categories,
                               remove=('headers', 'footers', 'quotes'))

data = []
labels = []

# Extract 20 samples per category
for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:20]
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])

print(f"Total samples collected: {len(data)}")

Total samples collected: 60


#**4. Step 2: Vectorization with CountVectorizer**

We use CountVectorizer to convert text into a Frequency-based Sparse Matrix. We also apply English stop-word removal to exclude common words like "the" or "a".

In [4]:
# Initialize CountVectorizer with English stop words removal
vectorizer = CountVectorizer(stop_words='english')

# Fit and transform the collected data
count_matrix = vectorizer.fit_transform(data)

print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")

Vocabulary size: 2342


# **5. Step 3: Implementing the Classification Function**

The classify_text function transforms the input sentence into the same vector space and calculates the cosine similarity against the 60 stored documents.Cosine Similarity FormulaThe similarity is calculated based on the angle between vectors:
$$\text{sim}(A, B) = \cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|}$$


In [5]:
def classify_text(input_text):
    # Transform input text (do NOT use fit_transform here)
    input_vec = vectorizer.transform([input_text])

    # Calculate cosine similarity
    sim = cosine_similarity(input_vec, count_matrix)

    # Find the index with the highest similarity score
    best_idx = np.argmax(sim)

    return labels[best_idx], sim[0][best_idx]

# **6. Step 4: Testing the Model**

We test the classifier with sample sentences representing the chosen topics .

In [6]:
test_sentences = [
    "The rocket launched into orbit.",
    "A new 3D rendering technique for graphics.",
    "Theological discussions on faith and god."
]

for s in test_sentences:
    cat, score = classify_text(s)
    print(f"Sentence: {s[:30]}... | Prediction: {cat} | Similarity: {score:.4f}")

Sentence: The rocket launched into orbit... | Prediction: sci.space | Similarity: 0.0870
Sentence: A new 3D rendering technique f... | Prediction: comp.graphics | Similarity: 0.1952
Sentence: Theological discussions on fai... | Prediction: talk.religion.misc | Similarity: 0.3430


# **7. Discussion and Analysis**

**Q1. Why does similarity sometimes result in 0.0000?**

If you input a sentence like "Exploring the mars with a robotic rover", the similarity might be 0. This occurs because CountVectorizer relies on exact word matches. If the specific words in your input (like "rover" or "mars") were not present in the 60 sampled training documents, the resulting vector for the input sentence will contain only zeros, making the dot product (and thus the similarity) zero.

**Q2. How to Improve Performance?**

To increase accuracy and similarity scores, one could try the following:



*   **Increase Sample Size:** Increasing from 20 to 100 samples per topic expands the vocabulary, making it more likely to find matching words.
*   **Use TfidfVectorizer:** Unlike CountVectorizer, TF-IDF penalizes common words and gives more weight to unique, meaningful terms.
*   **N-gram Range:** Setting ngram_range=(1, 2) allows the model to recognize phrases (bi-grams) like "space shuttle" rather than just individual words.










